# Notebook 03: Supervised Job Role Classification
**Project:** JobLens Vietnam - Labor Market Analysis & Skill Trends  
**Course:** Introduction to Data Science  

### Objective:
1. Formulate a multi-class text classification problem predicting `job_role_label` from `job_description`.
2. Address class imbalance across 8 job role categories.
3. Apply stratified 80/20 train/test split (random seed = 42).
4. Train and compare 5 distinct algorithm families on SBERT embeddings:
   - Logistic Regression (Balanced)
   - Linear Support Vector Machine (Linear SVM)
   - K-Nearest Neighbors (KNN with Cosine metric)
   - Random Forest Classifier
   - XGBoost Classifier
5. Evaluate models using Accuracy, Precision, Recall, Macro-F1 (primary metric), and Weighted-F1.
6. Display Confusion Matrix and analyze misclassifications.

In [ ]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

PROJECT_ROOT = Path('.').resolve().parent if Path('.').resolve().name == 'notebooks' else Path('.').resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import src.config as config
from src.models.classification import run_classification_experiments

## 1. Load Classification Dataset and Embeddings
We load `classification_data.csv` and slice the corresponding SBERT embedding matrix using aligned `unique_job_id` keys.

In [ ]:
df_cls = pd.read_csv(config.PROCESSED_DATA_DIR / "classification_data.csv")
sbert_all = np.load(config.EMBEDDINGS_DIR / "sbert_embeddings.npy")
cleaned_df = pd.read_csv(config.PROCESSED_DATA_DIR / "cleaned_jobs.csv")

# Map unique_job_id to embedding row index
id_to_idx = {uid: idx for idx, uid in enumerate(cleaned_df['unique_job_id'])}
cls_indices = [id_to_idx[uid] for uid in df_cls['unique_job_id']]
X_cls = sbert_all[cls_indices]
y_cls = df_cls['job_role_label']

print(f"Classification samples: {X_cls.shape[0]}, Feature dimension: {X_cls.shape[1]}")
print("\nClass Distribution:")
print(y_cls.value_counts())

## 2. Execute Model Training and Evaluation
We execute all 5 models. Class weighting is incorporated to prevent majority classes (e.g., Software Engineer) from dominating specialized roles (Data Scientist, AI Engineer).

In [ ]:
results_df, best_model, metadata = run_classification_experiments(X_cls, y_cls)
print("=== Classification Model Comparison Table ===")
display(results_df) if 'display' in dir() else print(results_df)

## 3. Detailed Classification Report and Confusion Matrix
Inspect per-class precision, recall, and F1 scores for the top-performing model.

In [ ]:
cr_path = config.CLASSIFICATION_OUTPUTS / 'metrics' / 'classification_report.csv'
if cr_path.exists():
    cr_df = pd.read_csv(cr_path)
    print("=== Per-Class Classification Report (Best Model) ===")
    display(cr_df) if 'display' in dir() else print(cr_df.to_string(index=False))

# Display Confusion Matrix Plot
cm_path = config.CLASSIFICATION_OUTPUTS / 'plots' / 'confusion_matrix.png'
if cm_path.exists():
    from IPython.display import Image, display as ipy_display
    ipy_display(Image(filename=str(cm_path)))